In [0]:
# creating silver layer which includes all 10 files

In [0]:
from pyspark.sql.functions import (col,trim,to_date,upper,lower,to_timestamp)


In [0]:

silver_customers = (
    spark.table("bronze_customers")

    # Remove records without primary key
    .filter(col("CustomerKey").isNotNull())

    # Remove duplicate customers
    .dropDuplicates(["CustomerKey"])

    # Clean text columns
    .withColumn("FirstName", trim(col("FirstName")))
    .withColumn("LastName", trim(col("LastName")))
    .withColumn("EmailAddress", lower(trim(col("EmailAddress"))))

    # Standardize gender
    .withColumn(
        "Gender",
        upper(trim(col("Gender")))
    )

    # Standardize marital status
    .withColumn(
        "MaritalStatus",
        upper(trim(col("MaritalStatus")))
    )
)

In [0]:
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

In [0]:
# 2. Product Categories

silver_product_categories = (
    spark.table("bronze_product_categories")

    .filter(col("ProductCategoryKey").isNotNull())

    .dropDuplicates(["ProductCategoryKey"])

    .withColumn(
        "CategoryName",
        trim(col("CategoryName"))
    )
)

In [0]:
silver_product_categories.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_product_categories")

In [0]:
# 3. Product Subcategories

silver_product_subcategories = (
    spark.table("bronze_product_subcategories")

    .filter(col("ProductSubcategoryKey").isNotNull())

    .dropDuplicates(["ProductSubcategoryKey"])

    .withColumn(
        "SubcategoryName",
        trim(col("SubcategoryName"))
    )
)

In [0]:
silver_product_subcategories.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_product_subcategories")

In [0]:
# 4. Products

silver_products = (
    spark.table("bronze_products")

    # Primary key validation
    .filter(col("ProductKey").isNotNull())

    # Remove duplicate products
    .dropDuplicates(["ProductKey"])

    # Clean text
    .withColumn(
        "ProductName",
        trim(col("ProductName"))
    )
    .withColumn(
        "ProductColor",
        trim(col("ProductColor"))
    )

    # Remove invalid prices
    .filter(
        (col("ProductCost").isNull()) |
        (col("ProductCost") >= 0)
    )
    .filter(
        (col("ProductPrice").isNull()) |
        (col("ProductPrice") >= 0)
    )
)

In [0]:

silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_products")

In [0]:
# 5. Calendar

silver_calendar = (
    spark.table("bronze_calendar")

    .filter(col("Date").isNotNull())

    .dropDuplicates(["Date"])

    .withColumn(
        "Date",
        to_date(col("Date"))
    )
)

In [0]:
silver_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_calendar")

In [0]:
# 6. Territories

silver_territories = (
    spark.table("bronze_territories")

    .filter(col("SalesTerritoryKey").isNotNull())

    .dropDuplicates(["SalesTerritoryKey"])

    .withColumn(
        "Region",
        trim(col("Region"))
    )

    .withColumn(
        "Country",
        trim(col("Country"))
    )

    .withColumn(
        "Continent",
        trim(col("Continent"))
    )
)

In [0]:
silver_territories.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_territories")

In [0]:
# 7. Returns

silver_returns = (
    spark.table("bronze_returns")

    # Required keys
    .filter(col("ProductKey").isNotNull())
    .filter(col("TerritoryKey").isNotNull())

    # Convert date
    .withColumn(
        "ReturnDate",
        to_date(col("ReturnDate"))
    )

    # Valid return quantity
    .filter(col("ReturnQuantity") > 0)

    # Remove duplicates
    .dropDuplicates()
)

In [0]:
silver_returns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_returns")

In [0]:
# 8. Sales 2015

silver_sales_2015 = (
    spark.table("bronze_sales_2015")

    # Required keys
    .filter(col("ProductKey").isNotNull())
    .filter(col("CustomerKey").isNotNull())

    # Clean date
    .withColumn(
        "OrderDate",
        to_date(col("OrderDate"))
    )

    # Valid transaction values
    .filter(col("OrderQuantity") > 0)

    # Remove duplicates
    .dropDuplicates()
)

In [0]:
silver_sales_2015.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales_2015")

In [0]:
# 9. Sales 2016

silver_sales_2016 = (
    spark.table("bronze_sales_2016")

    .filter(col("ProductKey").isNotNull())
    .filter(col("CustomerKey").isNotNull())

    .withColumn(
        "OrderDate",
        to_date(col("OrderDate"))
    )

    .filter(col("OrderQuantity") > 0)

    .dropDuplicates()
)

In [0]:
silver_sales_2016.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales_2016")

In [0]:
# 10. Sales 2017

silver_sales_2017 = (
    spark.table("bronze_sales_2017")

    .filter(col("ProductKey").isNotNull())
    .filter(col("CustomerKey").isNotNull())

    .withColumn(
        "OrderDate",
        to_date(col("OrderDate"))
    )

    .filter(col("OrderQuantity") > 0)

    .dropDuplicates()
)

In [0]:
silver_sales_2017.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales_2017")

In [0]:
# 11. Combine all Sales
silver_sales = (
    silver_sales_2015
    .unionByName(silver_sales_2016, allowMissingColumns=True)
    .unionByName(silver_sales_2017, allowMissingColumns=True)
)
# Remove duplicates:
silver_sales = silver_sales.dropDuplicates()

In [0]:
silver_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales")

In [0]:
silver_tables = [
    "silver_calendar",
    "silver_customers",
    "silver_product_categories",
    "silver_product_subcategories",
    "silver_products",
    "silver_returns",
    "silver_sales",
    "silver_territories"
]

for table in silver_tables:
    df = spark.table(table)

    print(
        f"{table}: {df.count()} rows, {len(df.columns)} columns"
    )

silver_calendar: 912 rows, 2 columns
silver_customers: 18148 rows, 14 columns
silver_product_categories: 4 rows, 3 columns
silver_product_subcategories: 37 rows, 4 columns
silver_products: 293 rows, 12 columns
silver_returns: 1809 rows, 5 columns
silver_sales: 56046 rows, 9 columns
silver_territories: 10 rows, 5 columns
